# Notebook 02 â€” Semantic Segmentation (SegFormer-B2)

Fine-tunes SegFormer-B2 on ADE20K + NYUv2 + custom staircase data. Exports ONNX for Jetson.

## Cell 02-01 | Mount & Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch, subprocess, sys

BASE    = '/content/drive/MyDrive/ARGUS'
DS      = f'{BASE}/datasets'
MODELS  = f'{BASE}/models'
CKPTS   = f'{BASE}/checkpoints'
EXPORTS = f'{BASE}/exports'
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'albumentations', 'huggingface_hub', 'onnx', 'onnxruntime'])
print(f'Device: {DEVICE}')

## Cell 02-02 | Download ADE20K Dataset

In [ ]:
import subprocess, os, zipfile, tarfile as _tarfile

def _safe_dl(url, dest, min_mb=None):
    """wget -c resume. If file exists but too small, delete and restart."""
    if os.path.exists(dest):
        mb = os.path.getsize(dest) / 1e6
        if min_mb and mb < min_mb * 0.90:
            print(f"  Partial file {os.path.basename(dest)} ({mb:.0f}/{min_mb} MB) — removing")
            os.remove(dest)
        else:
            print(f"  ✓ {os.path.basename(dest)} ({mb:.0f} MB)")
            return True
    print(f"  Downloading {os.path.basename(dest)} ...")
    r = subprocess.run(["wget", "-c", "--show-progress", "-O", dest, url])
    return r.returncode == 0

def _verify(path):
    """Return True if zip/tar is intact, delete it and return False if corrupt."""
    try:
        if path.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                bad = z.testzip()
                if bad: raise ValueError(bad)
        elif path.endswith((".tar.gz", ".tgz", ".tar")):
            with _tarfile.open(path) as t:
                t.getmembers()
        return True
    except Exception as e:
        print(f"  Corrupt archive ({e}) — removing for clean re-download")
        os.remove(path)
        return False

def _extract(path, dest):
    print(f"  Extracting {os.path.basename(path)} ...")
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z: z.extractall(dest)
    else:
        subprocess.run(["tar", "-xf", path, "-C", dest], check=True)


ADE_DIR   = f'{DS}/ade20k'
ADE_ZIP   = f'{ADE_DIR}/ADEChallengeData2016.zip'
ADE_URL   = 'http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip'
ADE_FLAG  = f'{ADE_DIR}/ade20k_done.flag'
os.makedirs(ADE_DIR, exist_ok=True)

if os.path.exists(ADE_FLAG):
    print('✓ ADE20K already extracted')
else:
    ok = _safe_dl(ADE_URL, ADE_ZIP, min_mb=800)
    if ok:
        if not _verify(ADE_ZIP):
            print('Re-run this cell to re-download the corrupt archive')
        else:
            _extract(ADE_ZIP, ADE_DIR)
            open(ADE_FLAG, 'w').close()
            print('✅ ADE20K ready')
    else:
        print('⚠ Download failed — re-run this cell to resume')
os.system(f'du -sh {ADE_DIR}')

## Cell 02-03 | Download NYUv2 Depth Dataset

In [ ]:
import subprocess, os, zipfile, tarfile as _tarfile

def _safe_dl(url, dest, min_mb=None):
    """wget -c resume. If file exists but too small, delete and restart."""
    if os.path.exists(dest):
        mb = os.path.getsize(dest) / 1e6
        if min_mb and mb < min_mb * 0.90:
            print(f"  Partial file {os.path.basename(dest)} ({mb:.0f}/{min_mb} MB) — removing")
            os.remove(dest)
        else:
            print(f"  ✓ {os.path.basename(dest)} ({mb:.0f} MB)")
            return True
    print(f"  Downloading {os.path.basename(dest)} ...")
    r = subprocess.run(["wget", "-c", "--show-progress", "-O", dest, url])
    return r.returncode == 0

def _verify(path):
    """Return True if zip/tar is intact, delete it and return False if corrupt."""
    try:
        if path.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                bad = z.testzip()
                if bad: raise ValueError(bad)
        elif path.endswith((".tar.gz", ".tgz", ".tar")):
            with _tarfile.open(path) as t:
                t.getmembers()
        return True
    except Exception as e:
        print(f"  Corrupt archive ({e}) — removing for clean re-download")
        os.remove(path)
        return False

def _extract(path, dest):
    print(f"  Extracting {os.path.basename(path)} ...")
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z: z.extractall(dest)
    else:
        subprocess.run(["tar", "-xf", path, "-C", dest], check=True)


NYU_DIR  = f'{DS}/nyuv2'
NYU_FILE = f'{NYU_DIR}/nyu_depth_v2_labeled.mat'
NYU_URL  = 'http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat'
os.makedirs(NYU_DIR, exist_ok=True)

# .mat file is ~2.8 GB
ok = _safe_dl(NYU_URL, NYU_FILE, min_mb=2600)
if ok:
    print(f'✅ NYUv2 ready ({os.path.getsize(NYU_FILE)/1e9:.2f} GB)')
else:
    print('⚠ Download incomplete — re-run this cell to resume')

## Cell 02-04 | ARGUS Custom Class Mapping

In [ ]:
ARGUS_CLASSES = {
    0: 'background',
    1: 'navigable_floor',
    2: 'wall',
    3: 'door_closed',
    4: 'door_open',
    5: 'stairs_up',
    6: 'stairs_down',
    7: 'person',
    8: 'furniture',
    9: 'clutter',
}

NUM_CLASSES = len(ARGUS_CLASSES)
print(f'ARGUS uses {NUM_CLASSES} semantic classes:')
for k, v in ARGUS_CLASSES.items():
    danger = '  SAFETY CRITICAL' if 'stairs' in v else ''
    print(f'  {k:2d}: {v}{danger}')

ADE_TO_ARGUS = {
    4: 1, 30: 1, 55: 1,
    1: 2, 14: 2,
    15: 3,
    54: 5,
    13: 7,
    8: 8, 10: 8, 19: 8, 24: 8, 34: 8,
}

def ade_to_argus_mask(ade_mask):
    import numpy as np
    argus_mask = np.zeros_like(ade_mask, dtype=np.uint8)
    for ade_cls, argus_cls in ADE_TO_ARGUS.items():
        argus_mask[ade_mask == ade_cls] = argus_cls
    return argus_mask

print('Class mapping defined')

## Cell 02-05 | Segmentation Dataset Class

In [ ]:
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from PIL import Image
import numpy as np, torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = 512

TRAIN_TRANSFORM = A.Compose([
    A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.5, 2.0)),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, p=0.5),
    A.GaussianBlur(p=0.1),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

VAL_TRANSFORM = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class ADE20KARGUSDataset(Dataset):
    def __init__(self, ade_root, split='training', transform=None):
        self.img_dir   = f'{ade_root}/ADEChallengeData2016/images/{split}'
        self.ann_dir   = f'{ade_root}/ADEChallengeData2016/annotations/{split}'
        self.transform = transform
        self.images    = sorted([f for f in os.listdir(self.img_dir) if f.endswith('.jpg')])
        print(f'ADE20K {split}: {len(self.images)} images')

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        ann_name = img_name.replace('.jpg', '.png')
        image    = np.array(Image.open(f'{self.img_dir}/{img_name}').convert('RGB'))
        mask     = np.array(Image.open(f'{self.ann_dir}/{ann_name}'))
        mask     = ade_to_argus_mask(mask)
        if self.transform:
            out = self.transform(image=image, mask=mask)
            return out['image'], out['mask'].long()
        return image, mask

class CustomSegDataset(Dataset):
    # Custom staircase/door annotations with ARGUS class masks (0-9)
    def __init__(self, root, transform=None):
        self.img_dir   = f'{root}/images'
        self.mask_dir  = f'{root}/masks'
        self.transform = transform
        self.images    = sorted([f for f in os.listdir(self.img_dir)
                                  if f.endswith(('.jpg', '.png'))]
                                ) if os.path.exists(self.img_dir) else []
        print(f'Custom seg data: {len(self.images)} images')

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        name  = self.images[idx]
        image = np.array(Image.open(f'{self.img_dir}/{name}').convert('RGB'))
        mask  = np.array(Image.open(f'{self.mask_dir}/{name}').convert('L'))
        if self.transform:
            out = self.transform(image=image, mask=mask)
            return out['image'], out['mask'].long()
        return image, mask

print('Dataset classes defined')

## Cell 02-06 | Load SegFormer-B2 & Fine-Tune

In [ ]:
from transformers import SegformerForSemanticSegmentation
from torch.optim import AdamW
from torch.optim.lr_scheduler import PolynomialLR
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset
from tqdm import tqdm
import time, shutil

CKPT_DIR = f'{CKPTS}/segformer_b2'
LATEST   = f'{CKPT_DIR}/latest.pth'
os.makedirs(CKPT_DIR, exist_ok=True)

ADE_ROOT   = f'{DS}/ade20k'
CUSTOM_SEG = f'{DS}/custom_seg'
train_ade    = ADE20KARGUSDataset(ADE_ROOT, 'training',   TRAIN_TRANSFORM)
val_ade      = ADE20KARGUSDataset(ADE_ROOT, 'validation', VAL_TRANSFORM)
custom_train = CustomSegDataset(CUSTOM_SEG, TRAIN_TRANSFORM)
train_ds     = ConcatDataset([train_ade] + [custom_train] * 3)
print(f'Total training samples: {len(train_ds)}')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=8, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ade,  batch_size=32, shuffle=False, num_workers=8, pin_memory=True)

EPOCHS    = 40
weights   = torch.ones(NUM_CLASSES).to(DEVICE)
weights[5] = 5.0; weights[6] = 5.0; weights[4] = 2.0
criterion = torch.nn.CrossEntropyLoss(weight=weights, ignore_index=255)

def _make_model():
    m = SegformerForSemanticSegmentation.from_pretrained(
        'nvidia/segformer-b2-finetuned-ade-512-512',
        num_labels=NUM_CLASSES, id2label=ARGUS_CLASSES,
        label2id={v: k for k, v in ARGUS_CLASSES.items()},
        ignore_mismatched_sizes=True,
    ).to(DEVICE)
    return m

def _make_optim(m):
    return AdamW([
        {'params': m.segformer.parameters(),    'lr': 6e-5},
        {'params': m.decode_head.parameters(),  'lr': 6e-4},
    ], weight_decay=0.01)

model     = _make_model()
optimizer = _make_optim(model)
scheduler = PolynomialLR(optimizer, total_iters=EPOCHS * len(train_loader), power=0.9)
scaler    = torch.cuda.amp.GradScaler()

# ── Resume ───────────────────────────────────────────────────────────────────
start_epoch = 0
best_miou   = 0.0
if os.path.exists(LATEST):
    try:
        state = torch.load(LATEST, map_location=DEVICE)
        model.load_state_dict(state['model_state'])
        optimizer.load_state_dict(state['optimizer_state'])
        scheduler.load_state_dict(state['scheduler_state'])
        scaler.load_state_dict(state['scaler_state'])
        start_epoch = state['epoch']
        best_miou   = state.get('best_miou', 0.0)
        print(f'Resumed from epoch {start_epoch}/{EPOCHS} | best_miou so far: {best_miou:.4f}')
    except Exception as e:
        print(f'Checkpoint corrupt ({e}) — starting from scratch')
        os.remove(LATEST)
        model     = _make_model()
        optimizer = _make_optim(model)
        scheduler = PolynomialLR(optimizer, total_iters=EPOCHS * len(train_loader), power=0.9)
        scaler    = torch.cuda.amp.GradScaler()
        start_epoch = 0
        best_miou   = 0.0

if start_epoch == 0:
    print('SegFormer-B2 loaded — starting from epoch 0')

def compute_miou(preds, labels, nc):
    ious = []
    for c in range(nc):
        inter = ((preds == c) & (labels == c)).sum().float()
        union = ((preds == c) | (labels == c)).sum().float()
        if union > 0: ious.append((inter / union).item())
    return sum(ious) / len(ious) if ious else 0.0

if start_epoch >= EPOCHS:
    print(f'Training already complete ({EPOCHS} epochs). Best mIoU: {best_miou:.4f}')
else:
    t_train_start = time.time()
    for epoch in range(start_epoch, EPOCHS):
        model.train(); train_loss = 0.0; t_ep = time.time()
        for images, masks in tqdm(train_loader,
                                  desc=f'Train Ep {epoch+1}/{EPOCHS}',
                                  unit='batch', dynamic_ncols=True):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                logits = F.interpolate(model(pixel_values=images).logits,
                                       size=masks.shape[-2:], mode='bilinear', align_corners=False)
                loss   = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            train_loss += loss.item()

        model.eval(); val_miou = []
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f'Val Ep {epoch+1}/{EPOCHS}',
                                      unit='batch', dynamic_ncols=True):
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                with torch.cuda.amp.autocast():
                    logits = F.interpolate(model(pixel_values=images).logits,
                                           size=masks.shape[-2:], mode='bilinear', align_corners=False)
                val_miou.append(compute_miou(logits.argmax(dim=1), masks, NUM_CLASSES))

        avg_miou  = sum(val_miou) / len(val_miou)
        avg_loss  = train_loss / len(train_loader)
        ep_t      = time.time() - t_ep
        tot       = time.time() - t_train_start
        eta       = ep_t * (EPOCHS - epoch - 1)
        print(f'Epoch {epoch+1}/{EPOCHS} | loss={avg_loss:.4f} | mIoU={avg_miou:.4f} | '
              f'ep={ep_t/60:.1f}m | elapsed={tot/3600:.2f}h | eta={eta/3600:.1f}h')

        # Save full state every epoch
        if avg_miou > best_miou:
            best_miou = avg_miou
            model.save_pretrained(f'{MODELS}/segmentation/segformer_b2_argus')
            print(f'  New best mIoU: {best_miou:.4f} — model saved')

        torch.save({
            'epoch'          : epoch + 1,
            'model_state'    : model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'scaler_state'   : scaler.state_dict(),
            'best_miou'      : best_miou,
        }, LATEST)
        if (epoch + 1) % 10 == 0:
            shutil.copy(LATEST, f'{CKPT_DIR}/epoch_{epoch+1}.pth')

print(f'Training complete. Best mIoU: {best_miou:.4f}')


## Cell 02-07 | Export SegFormer to ONNX

In [ ]:
model.eval()
H, W      = 512, 512
dummy_in  = torch.randn(1, 3, H, W).to(DEVICE)
ONNX_PATH = f'{EXPORTS}/tensorrt/segformer_b2_512x512.onnx'

class SegFormerExport(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.model = m
    def forward(self, x):
        return F.interpolate(self.model(pixel_values=x).logits, size=(H, W),
                             mode='bilinear', align_corners=False)

with torch.no_grad():
    torch.onnx.export(SegFormerExport(model), dummy_in, ONNX_PATH,
        input_names=['image'], output_names=['segmentation'],
        dynamic_axes={'image': {0: 'batch'}, 'segmentation': {0: 'batch'}},
        opset_version=17)
print(f'ONNX exported: {ONNX_PATH}')
print(f'Size: {os.path.getsize(ONNX_PATH)/1e6:.1f} MB')